# 🎬 CineMatch — Live Deployment

Run each cell in order. After the last cell you get a public URL anyone can visit.

**Before running:** Fill in your tokens in Cell 4 and Cell 6.

**Keep this tab open** — closing it stops the server.

In [ ]:
# Cell 1: Install dependencies
!pip install -q flask flask-cors python-dotenv requests \
    pandas numpy scikit-learn scipy joblib \
    sentence-transformers nltk huggingface_hub pyngrok

In [ ]:
# Cell 2: Clone the GitHub repo
import os
if os.path.exists('cinematch'):
    !cd cinematch && git pull
else:
    !git clone https://github.com/Abhinand0708/cinematch.git
os.chdir('cinematch')
print('Working directory:', os.getcwd())

In [ ]:
# Cell 3: Download model files from Hugging Face Hub
import os
from huggingface_hub import hf_hub_download

HF_REPO_ID = 'abhinand8386/cinematch-models'

FILES = [
    ('models/final_hybrid_config.pkl',           'models'),
    ('models/hybrid_config.pkl',                 'models'),
    ('models/movie_embeddings.npy',              'models'),
    ('models/tfidf_vectorizer.pkl',              'models'),
    ('models/tfidf_matrix.npz',                  'models'),
    ('models/genre_sets.pkl',                    'models'),
    ('dataset/recommendation_model_dataset.csv', 'dataset'),
    ('dataset/final_movie_dataset.csv',          'dataset'),
]

os.makedirs('models',  exist_ok=True)
os.makedirs('dataset', exist_ok=True)

for filename, dest_dir in FILES:
    local_name = os.path.basename(filename)
    dest_path  = os.path.join(dest_dir, local_name)
    if os.path.exists(dest_path):
        print(f'  Already exists: {local_name}')
        continue
    print(f'  Downloading {local_name}...', end=' ', flush=True)
    hf_hub_download(repo_id=HF_REPO_ID, filename=filename,
                    repo_type='dataset', local_dir='.')
    print('done')

print('All model files ready!')

In [ ]:
# Cell 4: Set your TMDB token
# Get your token from: https://www.themoviedb.org/settings/api
import os

os.environ['TMDB_ACCESS_TOKEN']      = 'PASTE_YOUR_TMDB_TOKEN_HERE'
os.environ['KMP_DUPLICATE_LIB_OK']   = 'TRUE'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['FLASK_DEBUG']            = '0'

print('Environment variables set.')

In [ ]:
# Cell 5: Download NLTK data
import nltk
for resource in ['punkt','punkt_tab','stopwords','wordnet','omw-1.4','averaged_perceptron_tagger_eng']:
    nltk.download(resource, quiet=True)
print('NLTK data ready.')

In [ ]:
# Cell 6: Start Flask + ngrok
# Get your ngrok token from: https://dashboard.ngrok.com/get-started/your-authtoken
import sys, os, threading

NGROK_TOKEN = 'PASTE_YOUR_NGROK_TOKEN_HERE'

sys.path.insert(0, os.path.join(os.getcwd(), 'backend'))

print('Loading ML models (takes ~30 seconds)...')
import model_loader
models = model_loader.load_all()
model_loader._MODELS = models
print('Models loaded!')

from app import app
app.config['DEBUG'] = False

def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

threading.Thread(target=run_flask, daemon=True).start()
print('Flask started on port 5000')

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(5000)

print()
print('=' * 60)
print('CineMatch is LIVE!')
print(f'Public URL: {public_url.public_url}')
print('=' * 60)
print('Share this URL with anyone. Keep this tab open.')